### Esercitazione - E04 - Generative Adversarial Networks

Scarica un dataset di immagini reali, costruisci la tua GAN e genera nuove immagini!

* Usa il dataset [Oxford 102 Flowers](https://pytorch.org/vision/0.16/generated/torchvision.datasets.Flowers102.html#torchvision.datasets.Flowers102).

* Le immagini di questo dataset in genere hanno shape (3, 128, 128), ma per velocizzare il training puoi usare anche la dimensione (3, 64, 64).

* Costruisci una GAN basata su reti convoluzionali!

* Nei casi reali, le GAN sono molto sensibili al settaggio degli iperparametri e alla costruzione dell'architettura. Bisogna giocare un po' con gli iperparametri per essere sicuri che funzioni!

⚠️ Puoi utilizzare blocchi di codice che abbiamo scritto nei notebook precedenti!

In [1]:
# Import

import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision import datasets
from torchvision.utils import save_image
import matplotlib.pyplot as plt

In [2]:
# Impostiamo gli hyperparametri
batch_size = 128
num_epoch = 100
z_dimension = 100  # dimensione del vettore di rumore in input al generatore
lr = 1e-3
epochs = 100

# Trasformazione delle immagini
def to_img(x):
    out = 0.5 * (x + 1)
    out = out.clamp(0, 1)
    out = out.view(-1, 3, 64, 64)
    return out

img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((64, 64)),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# Scarichiamo e prepariamo il dataset

train_ds = datasets.Flowers102("./dataset", split="train", transform=img_transform, download=True)
test_ds = datasets.Flowers102("./dataset", split="test", transform=img_transform, download=True)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [5]:
if not os.path.exists('./dc_img'):
    os.mkdir('./dc_img')


# Definiamo la classe dei modelli nella GAN:
# Discriminatore
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2),  # batch, 32, 64, 64
            nn.LeakyReLU(0.2, True),
            nn.AvgPool2d(2, stride=2),  # batch, 32, 32, 32
            )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 5, padding=2),  # batch, 64, 32, 32
            nn.LeakyReLU(0.2, True),
            nn.AvgPool2d(2, stride=2)  # batch, 64, 16, 16
        )
        self.fc = nn.Sequential(
            nn.Linear(64*16*16, 1024), # [16384, 1024]
            nn.LeakyReLU(0.2, True),
            nn.Linear(1024, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        '''
        x: batch, 3, 64, 64
        '''
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Generatore
class Generator(nn.Module):
    def __init__(self, input_size, num_feature):
        super(Generator, self).__init__()
        self.fc = nn.Linear(input_size, num_feature*3)  # batch, 3*56*56
        self.br = nn.Sequential(
            nn.BatchNorm2d(3),
            nn.ReLU(True)
        )
        self.downsample1 = nn.Sequential(
            nn.Conv2d(3, 50, 3, stride=1, padding=1),  # batch, 50, 56, 56
            nn.BatchNorm2d(50),
            nn.ReLU(True)
        )
        self.downsample2 = nn.Sequential(
            nn.Conv2d(50, 25, 3, stride=1, padding=1),  # batch, 25, 56, 56
            nn.BatchNorm2d(25),
            nn.ReLU(True)
        )
        self.downsample3 = nn.Sequential(
            nn.Conv2d(25, 3, 2, stride=2),  # batch, 3, 28, 28
            nn.Tanh()
        )

    def forward(self, x):
        x = self.fc(x)
        x = x.view(x.size(0), 3, 56, 56)
        x = self.br(x)
        x = self.downsample1(x)
        x = self.downsample2(x)
        x = self.downsample3(x)
        return x

In [6]:
# Creiamo un'istanza del Discriminatore
D = Discriminator().to(DEVICE)

# Creiamo un'istanza del Generatore
G = Generator(z_dimension, 3136).to(DEVICE)

# Definiamo la loss function
criterion = nn.BCELoss()

# Definiamo gli ottimizzatori per il discriminatore e il generatore
d_optimizer = torch.optim.Adam(D.parameters(), lr=lr)
g_optimizer = torch.optim.Adam(G.parameters(), lr=lr)

In [7]:
# Alleniamo le nostre reti

def fit():

    for epoch in range(epochs):

        for i, (img, labels) in enumerate(train_dl):

            img = img.to(DEVICE)
            real_y = torch.ones(img.size(0)).to(DEVICE) #shape (1, ) -> lista etichette
            fake_y = torch.zeros(img.size(0)).to(DEVICE)

            real_out = D(img).squeeze(1)
            d_loss_real = criterion(real_out, real_y)

            z = torch.randn(img.size(0), z_dimension).to(DEVICE)
            fake_img = G(z)
            fake_out = D(fake_img).squeeze(1)

            d_loss_fake = criterion(fake_out, fake_y)

            d_loss = d_loss_real + d_loss_fake

            d_optimizer.zero_grad()
            d_loss.backward()
            d_optimizer.step()


            z = torch.randn(img.size(0), z_dimension).to(DEVICE)
            fake_img = G(z)
            out = D(fake_img).squeeze(1)

            g_loss = criterion(out, real_y)

            g_optimizer.zero_grad()
            g_loss.backward()
            g_optimizer.step()

        fake_images = to_img(fake_img.cpu().data)
        save_image(fake_images, './dc_img/fake_images-{}.png'.format(epoch+1))

torch.save(G.state_dict(), './generator.pth')
torch.save(D.state_dict(), './discriminator.pth')

In [8]:
fit()

RuntimeError: mat1 and mat2 shapes cannot be multiplied (128x3136 and 16384x1024)

In [ ]:
# Generiamo nuovi campioni